# Module 9: Linear regression

**Goal:** model how an outcome changes with other variables, fit that model **three ways** (formula, calculus/gradient descent, and library),
interpret its coefficients honestly, and check whether it's trustworthy.

**You will learn:** the line of best fit · least squares · gradient descent · multiple regression and the matrix form · interpreting coefficients ·
`statsmodels` and `scikit-learn` · residual diagnostics · $R^2$ and its traps

**Maths in this module:** slope and intercept · derivatives and the gradient · least squares · matrix form $\hat y=X\beta$ · $R^2$

> 📈 A regression is a *summary of an average relationship*, not a law of nature. Keep two questions in mind:
> **how well does it fit?** and **what is it (not) telling me?**

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

sns.set_theme(style="whitegrid", palette="colorblind")
students = pd.read_csv(DATA / "students.csv")
x = students["study_hours_week"]
y = students["exam_score"]

## 1. The idea: a line through a cloud

A **simple linear regression** models the *average* outcome as a straight line:

$$\hat y = b_0 + b_1 x$$

* $b_1$ is the **slope**: the change in predicted $y$ for one extra unit of $x$.
* $b_0$ is the **intercept**: the predicted $y$ when $x=0$ (sometimes meaningful, sometimes just an anchor).
* The **residual** $e_i=y_i-\hat y_i$ is the miss for student $i$.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(x, y, alpha=0.6)
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, 52 + 1.5 * xs, color="tab:red", label="a guess: 52 + 1.5·x")
ax.set_xlabel("Study hours per week"); ax.set_ylabel("Exam score"); ax.legend()
plt.show()

Infinitely many lines are possible. Which is **best**?

## 2. Least squares: defining "best"

Measure how bad a line is by the **mean squared error** (MSE) of its residuals:

$$L(b_0,b_1)=\frac1n\sum_{i=1}^{n}\big(y_i-b_0-b_1x_i\big)^2$$

Squaring makes misses positive and punishes big misses more. The **least-squares line** is the one with the smallest $L$. Let's see the loss as we vary the slope (re-fitting the best intercept each time):

In [ ]:
def mse_for_slope(b1):
    b0 = y.mean() - b1 * x.mean()            # best intercept for this slope
    return np.mean((y - b0 - b1 * x) ** 2)

slopes = np.linspace(0, 3, 121)
losses = [mse_for_slope(b) for b in slopes]

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot(slopes, losses)
best = slopes[int(np.argmin(losses))]
ax.axvline(best, color="tab:red", linestyle="--")
ax.set_xlabel("Slope $b_1$"); ax.set_ylabel("MSE"); ax.set_title(f"Loss is a bowl; the bottom is near {best:.2f}")
plt.show()

### The closed-form answer

Setting the derivatives of $L$ to zero (see below) gives formulas you can compute directly:

$$b_1=r\,\frac{s_y}{s_x},\qquad b_0=\bar y-b_1\bar x$$

The slope is the correlation, rescaled into $y$-units per $x$-unit. And **the line always passes through $(\bar x,\bar y)$.**

In [ ]:
r = x.corr(y)
b1 = r * y.std() / x.std()
b0 = y.mean() - b1 * x.mean()
print(f"b1 = {b1:.4f}, b0 = {b0:.3f}   |   np.polyfit says:", np.polyfit(x, y, 1).round(4))

> *Reading it:* students who study **one more hour per week** score about **1.5 points higher on average**. That's an association in this sample.
> (Module 8 told us to attach an interval; we'll do that in §5.)

## 🧮 3. Calculus view: derivatives and gradient descent

A **derivative** is the slope of a curve at a point, the instantaneous rate of change. Two rules cover almost everything we need:

* **Power rule:** $\dfrac{d}{dx}x^2=2x$
* **Chain rule:** $\dfrac{d}{dx}\,g(h(x))=g'(h(x))\cdot h'(x)$

At the bottom of a bowl the slope is **zero**; that's how minima are found. With two parameters we take **partial derivatives** (change one parameter, freeze the other) and stack them into the **gradient**. For the loss above, with residuals $e_i=y_i-b_0-b_1x_i$:

$$\frac{\partial L}{\partial b_0}=-\frac2n\sum e_i\qquad\frac{\partial L}{\partial b_1}=-\frac2n\sum e_i\,x_i$$

Setting both to zero yields the closed-form formulas above. When no closed form exists (most of machine learning), we use **gradient descent**: the gradient points *uphill*, so repeatedly step a little **downhill**:

$$b\;\leftarrow\;b-\eta\,\nabla L(b)\qquad(\eta=\text{learning rate})$$

In [ ]:
def gradient_descent(x, y, lr=0.1, n_iter=200):
    b0, b1 = 0.0, 0.0
    history = []
    for _ in range(n_iter):
        resid = y - (b0 + b1 * x)
        history.append(np.mean(resid ** 2))
        grad_b0 = -2 * np.mean(resid)
        grad_b1 = -2 * np.mean(resid * x)
        b0 -= lr * grad_b0
        b1 -= lr * grad_b1
    return b0, b1, history

# Standardise x first (z-score, Module 3): it makes the loss bowl round, so descent is fast and stable
z = ((x - x.mean()) / x.std()).to_numpy()
c0, c1, hist = gradient_descent(z, y.to_numpy(), lr=0.1, n_iter=200)

# Convert back to the original x units: y = c0 + c1*(x - xbar)/s  ->  slope c1/s
slope = c1 / x.std()
intercept = c0 - slope * x.mean()
print(f"gradient descent: b1 = {slope:.4f}, b0 = {intercept:.3f}   (closed form: {b1:.4f}, {b0:.3f})")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(hist); ax.set_yscale("log")
ax.set_xlabel("Iteration"); ax.set_ylabel("MSE (log scale)"); ax.set_title("Gradient descent walking down the bowl")
plt.show()

**The learning rate matters.** Too small: it crawls. Too big: it overshoots and **diverges** (the loss explodes instead of shrinking).
Feature scale matters too: with raw `x` (values up to 40) the largest *safe* learning rate is tiny, whereas standardised `x` tolerates big steps.

In [ ]:
print("standardised x:")
for lr in [0.01, 0.1, 1.05]:
    _, _, h = gradient_descent(z, y.to_numpy(), lr=lr, n_iter=50)
    print(f"  lr={lr:<6} MSE after 50 steps: {h[-1]:.3g}")

print("raw x (study hours):")
for lr in [0.005, 0.01]:
    _, _, h = gradient_descent(x.to_numpy(), y.to_numpy(), lr=lr, n_iter=50)
    print(f"  lr={lr:<6} MSE after 50 steps: {h[-1]:.3g}")

With standardised `x`, $\eta=0.1$ is nearly converged (MSE ≈ 53, the least-squares minimum) while 1.05 explodes. With raw `x`, even $\eta=0.005$ is still far from converged after 50 steps (the
stable range is below about 0.008), and $\eta=0.01$ diverges. **Standardise your features before gradient-based fitting.**

## 4. Multiple regression and the matrix form

Use several predictors at once:

$$\hat y=b_0+b_1x_1+b_2x_2+\dots+b_px_p$$

Stack the data in a matrix $X$ (a column of 1s for the intercept, then one column per predictor) and the coefficients in a vector $\beta$: then **all predictions** are $\hat y=X\beta$.
That's the dot-product-per-row idea from Module 3. The least-squares solution (the **normal equations**) is

$$\hat\beta=(X^\top X)^{-1}X^\top y$$

In [ ]:
X = np.column_stack([np.ones(len(students)), students["study_hours_week"], students["sleep_hours"], students["attendance_pct"]])
beta = np.linalg.solve(X.T @ X, X.T @ y.to_numpy())       # solve is more stable than inverting
print("normal equations:", beta.round(3))

feats = ["study_hours_week", "sleep_hours", "attendance_pct"]
sk = LinearRegression().fit(students[feats], y)
print("scikit-learn    :", np.r_[sk.intercept_, sk.coef_].round(3))

### `statsmodels`: coefficients *with* uncertainty

scikit-learn is built for prediction; **`statsmodels`** reports the statistics from Module 8 (standard errors, intervals, p-values). Its **formula** syntax reads like the model:
`outcome ~ predictor1 + predictor2`. Text columns become dummy (0/1) variables automatically.

In [ ]:
model = smf.ols("exam_score ~ study_hours_week + sleep_hours + attendance_pct + part_time_job", data=students).fit()

table = pd.DataFrame({"coef": model.params, "ci_low": model.conf_int()[0], "ci_high": model.conf_int()[1], "p": model.pvalues}).round(3)
table

In [ ]:
print(f"R² = {model.rsquared:.3f}   adjusted R² = {model.rsquared_adj:.3f}   residual sd = {math.sqrt(model.mse_resid):.2f} points")

### How to read the coefficients

* `study_hours_week` ≈ **1.58**: *holding sleep, attendance and job status fixed*, each extra weekly study hour goes with about 1.6 more points.
* `part_time_job[T.yes]` ≈ **−3.2**: comparing students who are otherwise alike, those with a job score about 3 points lower. The word **"holding … fixed"** is what makes multiple regression different from comparing raw averages.
* The **intercept** (≈ 15.6) is the prediction for a student with 0 study hours, 0 sleep, 0 attendance, and no job. That's far outside the data, so don't interpret it.

**Compare with Module 8:** the *raw* difference for the job was −2.9 with a 95% CI of (−6.1, 0.3), which included 0. After adjusting, it's −3.2 with a CI of about (−4.9, −1.4).
The estimate barely moved, but it's **more precise** because study, sleep and attendance soak up variation that was previously noise (residual sd fell from ~10.5 to ~5.8).

> ⚠️ That's a legitimate reason for the interval to narrow, but beware of *shopping* for a model until the p-value you want appears (Module 8, §7).
> Decide which variables belong in the model **from your question**, before you look at the results.

**Categorical predictors** use a reference group: `part_time_job[T.yes]` is the *difference of "yes" relative to "no"*.

## 5. Diagnostics: is a line even appropriate?

Regression assumes roughly: (1) a **linear** relationship, (2) residuals with similar spread everywhere (**constant variance**), (3) residuals that look **bell-shaped** and (4) **independent** observations. Plot the residuals:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].scatter(model.fittedvalues, model.resid, alpha=0.6)
axes[0].axhline(0, color="tab:red")
axes[0].set_xlabel("Fitted value"); axes[0].set_ylabel("Residual"); axes[0].set_title("Residuals vs fitted: want a shapeless cloud")
axes[1].hist(model.resid, bins=20, edgecolor="white")
axes[1].set_xlabel("Residual"); axes[1].set_title("Residuals: roughly bell-shaped?")
plt.tight_layout(); plt.show()

A **curved** residual pattern means the model is missing structure. Simulating a quadratic truth shows what that looks like and how to fix it:

In [ ]:
rng = np.random.default_rng(1)
xs = rng.uniform(-3, 3, 150)
curve = pd.DataFrame({"x": xs, "y": 2 + 0.8 * xs ** 2 + rng.normal(0, 0.8, 150)})

line_fit = smf.ols("y ~ x", curve).fit()
quad_fit = smf.ols("y ~ x + I(x**2)", curve).fit()          # I(...) lets us add an x² term

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].scatter(line_fit.fittedvalues, line_fit.resid, alpha=0.6); axes[0].axhline(0, color="tab:red")
axes[0].set_title(f"straight line: R² = {line_fit.rsquared:.2f}, residuals curve")
axes[1].scatter(quad_fit.fittedvalues, quad_fit.resid, alpha=0.6); axes[1].axhline(0, color="tab:red")
axes[1].set_title(f"with x²: R² = {quad_fit.rsquared:.2f}, residuals shapeless")
for ax in axes: ax.set_xlabel("Fitted"); ax.set_ylabel("Residual")
plt.tight_layout(); plt.show()

Fixes for nonlinearity: add terms like $x^2$, **transform** variables (e.g. `np.log`, Module 5), or use a different kind of model (Module 10).
A *funnel* shape (spread growing with the fitted value) suggests transforming $y$ (often with a log).

## 6. $R^2$: how much does the model explain?

$$R^2=1-\frac{\text{SSE}}{\text{SST}}=1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$$

It compares your model's squared errors with those of the "just predict the mean" baseline. 0 = no better than the mean; 1 = perfect.
It's **unitless**; **RMSE** (the typical miss, in $y$-units) is often more intuitive.

**Traps:**
* **$R^2$ never decreases when you add predictors**, even useless ones. Watch:

In [ ]:
rng = np.random.default_rng(0)
noise = pd.DataFrame(rng.normal(size=(len(students), 10)), columns=[f"noise{i}" for i in range(10)])
with_noise = pd.concat([students, noise], axis=1)

base = smf.ols("exam_score ~ study_hours_week + sleep_hours + attendance_pct", students).fit()
padded = smf.ols("exam_score ~ study_hours_week + sleep_hours + attendance_pct + " + " + ".join(noise.columns), with_noise).fit()

print(f"R²:          {base.rsquared:.3f} -> {padded.rsquared:.3f}   (went UP after adding 10 random columns!)")
print(f"adjusted R²: {base.rsquared_adj:.3f} -> {padded.rsquared_adj:.3f}   (penalises useless predictors)")

That is **overfitting** starting: the model learns quirks of *this* sample. The cure is to judge models on **data they haven't seen**, the subject of Module 10.

* **Extrapolation:** don't predict far outside the range of your data (the line may not continue).
* **Causation:** coefficients describe associations. "Holding other variables fixed" only covers variables *you included*.
* **Outliers and leverage:** a few extreme points can tilt the line. Look at the plot.
* **Multicollinearity:** if two predictors move together (e.g. height in cm and in inches) the model can't separate their effects, so coefficients become unstable with huge standard errors.

## 7. A model-building recipe

1. **Plot** first (scatter, correlations). Is a line plausible?
2. **Choose predictors from your question** (and write them down before fitting).
3. Fit with `statsmodels`; read **coefficients + intervals** in the units of the data.
4. Check **residual plots**.
5. Report fit (**RMSE**, $R^2$) *and* limits (sample, range, association ≠ causation).
6. In Module 10: test it on **new data**.

## Recap

* Least squares picks the line minimising squared residuals; closed form: $b_1=r\,s_y/s_x$, $b_0=\bar y-b_1\bar x$.
* Gradient descent follows $-\nabla L$ downhill; scale features and tune the learning rate.
* Matrix form $\hat y=X\beta$; coefficients mean "change in $\hat y$ per unit of $x$, **holding the others fixed**".
* Always check residuals; $R^2$ alone can mislead and always rises with added predictors.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** (a first model).